In [1]:
import jax

jax.config.update("jax_platform_name", "cpu")

import sys
from functools import partial

import jax.numpy as jnp
from jax import vmap

sys.path.append("..")
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import LinearLocator

from configuration.rte2d_settings import get_config

# from utils.integrate import leggauss
from utils.coord_generator import cartesian_product

In [ ]:
rte_config = get_config()
domain = dict(rte_config.mesh["domain"])
print(f"domian: {domain}")
kn = rte_config.model.knudsen_number
num_quads = rte_config.model.num_quads
fn_sigma_s = rte_config.model.coeff["scattering"]
fn_sigma_a = rte_config.model.coeff["absorption"]
fn_l = rte_config.model.bdy_cond["f_l"]
fn_r = rte_config.model.bdy_cond["f_r"]
fn_b = rte_config.model.bdy_cond["f_b"]
fn_t = rte_config.model.bdy_cond["f_t"]
source_fn = rte_config.model.source

num_x, num_y, num_theta = rte_config.model.grid_sizes
print(f"num_x: {num_x}, num_y: {num_y}, num_theta: {num_theta}")

(xmin, xmax), (vmin, vmax), (theta_min, theta_max) = (
    domain["x"],
    domain["y"],
    domain["theta"],
)
dx = (xmax - xmin) / num_x
dy = (vmax - vmin) / num_y
dtheta = (theta_max - theta_min) / num_theta
mesh_x = np.linspace(xmin, xmax, num_x + 1)
mesh_xc = np.linspace(
    xmin + dx / 2, xmax - dx / 2, num_x
)  # num_x pts in cell center (not include gohst points)
mesh_y = np.linspace(vmin, vmax, num_y + 1)
mesh_yc = np.linspace(
    vmin + dy / 2, vmax - dy / 2, num_y
)  # num_x pts in cell center (not include gohst points)
mesh_theta = np.linspace(theta_min, theta_max, num_theta * 4)
cos_dx = (np.cos(mesh_theta) / dx)[None, None, :]  # (1, 1, num_theta)
sin_dy = (np.sin(mesh_theta) / dy)[None, None, :]  # (1, 1, num_theta)

mesh_xc, mesh_yc, mesh_theta

In [ ]:
# compute the scattering and absorption coefficient at each cell center
mesh_xyc = cartesian_product(
    [mesh_xc[:, None], mesh_yc[:, None]]
)  # (num_x, num_y, 2)
X, Y = [d.squeeze() for d in jnp.split(mesh_xyc, 2, axis=-1)]  # (num_x, num_y)
# mat_fn_sigma_s = vmap(fn_sigma_s, in_axes=(0, 0))(
#     X.reshape(-1), Y.reshape(-1)
# ).reshape(num_x, num_y)  # (num_x, num_y)
mat_fn_sigma_s = vmap(vmap(fn_sigma_s, in_axes=(0, 0)), in_axes=(0, 0))(
    X, Y
)  # (num_x, num_y)
mat_fn_sigma_a = vmap(vmap(fn_sigma_a, in_axes=(0, 0)), in_axes=(0, 0))(X, Y)
tensor_fn_sigma_s = np.asarray(mat_fn_sigma_s)[..., None]
tensor_fn_sigma_a = np.asarray(mat_fn_sigma_a)[..., None]
tensor_fn_sigma_t = (
    tensor_fn_sigma_s + tensor_fn_sigma_a * kn**2
)  # (num_x, num_y, 1)

tensor_fn_sigma_s.shape, mesh_xyc.shape

In [4]:
weights = 2.0 * np.pi / (num_theta * 4 - 1) * np.ones(num_theta * 4)
weights[0] *= 0.5
weights[-1] *= 0.5

In [5]:
# compute the source term at each cell center and each angle
mesh_xyc_theta = cartesian_product(
    [mesh_xc[:, None], mesh_yc[:, None], mesh_theta[:, None]]
)
X, Y, Theta = [
    d.squeeze() for d in jnp.split(mesh_xyc_theta, 3, axis=-1)
]  # (num_x, num_y, num_theta)

tensor_source = vmap(
    vmap(vmap(source_fn, in_axes=(0, 0, 0)), in_axes=(0, 0, 0)),
    in_axes=(0, 0, 0),
)(X, Y, Theta)  # (num_x, num_y, num_theta)

In [6]:
tensor_f = np.zeros((num_x + 1, num_y + 1, num_theta * 4))

# compute the boundary condition
tensor_f[0, :, (mesh_theta <= 0.5 * np.pi) | (mesh_theta >= 1.5 * np.pi)] = (
    fn_l(mesh_y)[None, :]
)
tensor_f[-1, :, (mesh_theta > 0.5 * np.pi) & (mesh_theta < 1.5 * np.pi)] = (
    fn_r(mesh_y)[None, :]
)
tensor_f[:, 0, (mesh_theta <= np.pi)] = fn_b(mesh_x)[:, None]
tensor_f[:, -1, (mesh_theta > np.pi)] = fn_t(mesh_x)[:, None]

In [7]:
def collision(tensor_f, weights, tensor_fn_sigma_s, tensor_source, kn):
    tensor_fc = (
        tensor_f[:-1, :-1]
        + tensor_f[1:, :-1]
        + tensor_f[:-1, 1:]
        + tensor_f[1:, 1:]
    ) / 4
    avg = 0.5 / np.pi * tensor_fc @ weights[:, None]  # (num_x, num_y, 1)
    qc = (
        tensor_fn_sigma_s * avg + kn**2 * tensor_source
    )  # (num_x, num_y, num_theta)
    return qc

In [8]:
coll_op = partial(
    collision,
    weights=weights,
    tensor_fn_sigma_s=tensor_fn_sigma_s,
    tensor_source=tensor_source,
    kn=kn,
)


In [9]:
def source_iteration(tensor_f):
    F = tensor_f.copy()  # (num_x+1, num_y+1, num_theta*4)
    coeff11 = (cos_dx + sin_dy) * kn / 2 + tensor_fn_sigma_t / 4
    coeff12 = (cos_dx - sin_dy) * kn / 2 + tensor_fn_sigma_t / 4
    coeff21 = (-cos_dx + sin_dy) * kn / 2 + tensor_fn_sigma_t / 4
    coeff22 = (-cos_dx - sin_dy) * kn / 2 + tensor_fn_sigma_t / 4
    q = coll_op(F)  # (num_x, num_y, num_theta*4)
    for j in range(num_y):
        # 0 < theta < 0.5*np.pi
        for i in range(num_x):
            F[i + 1, j + 1, mesh_theta <= 0.5 * np.pi] = (
                q[i, j, mesh_theta <= 0.5 * np.pi]
                - coeff12[i, j, mesh_theta <= 0.5 * np.pi]
                * F[i + 1, j, mesh_theta <= 0.5 * np.pi]
                - coeff21[i, j, mesh_theta <= 0.5 * np.pi]
                * F[i, j + 1, mesh_theta <= 0.5 * np.pi]
                - coeff22[i, j, mesh_theta <= 0.5 * np.pi]
                * F[i, j, mesh_theta <= 0.5 * np.pi]
            ) / coeff11[i, j, mesh_theta <= 0.5 * np.pi]
        # 0.5*np.pi < theta < np.pi
        for i in range(-1, -num_x - 1, -1):
            F[
                i - 1, j + 1, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
            ] = (
                q[i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)]
                - coeff11[
                    i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
                ]
                * F[
                    i, j + 1, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
                ]
                - coeff12[
                    i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
                ]
                * F[i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)]
                - coeff22[
                    i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
                ]
                * F[
                    i - 1, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
                ]
            ) / coeff21[
                i, j, (mesh_theta > 0.5 * np.pi) & (mesh_theta < np.pi)
            ]

    for j in range(-1, -num_y - 1, -1):
        # np.pi < theta < 1.5*np.pi
        for i in range(-1, -num_x - 1, -1):
            F[
                i - 1, j - 1, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
            ] = (
                q[i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)]
                - coeff11[
                    i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
                ]
                * F[i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)]
                - coeff12[
                    i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
                ]
                * F[
                    i, j - 1, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
                ]
                - coeff21[
                    i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
                ]
                * F[
                    i - 1, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
                ]
            ) / coeff22[
                i, j, (mesh_theta > np.pi) & (mesh_theta < 1.5 * np.pi)
            ]
        # 1.5*np.pi < theta < 2*np.pi
        for i in range(num_x):
            F[i + 1, j - 1, mesh_theta >= 1.5 * np.pi] = (
                q[i, j, mesh_theta >= 1.5 * np.pi]
                - coeff11[i, j, mesh_theta >= 1.5 * np.pi]
                * F[i + 1, j, mesh_theta >= 1.5 * np.pi]
                - coeff21[i, j, mesh_theta >= 1.5 * np.pi]
                * F[i, j, mesh_theta >= 1.5 * np.pi]
                - coeff22[i, j, mesh_theta >= 1.5 * np.pi]
                * F[i, j - 1, mesh_theta >= 1.5 * np.pi]
            ) / coeff12[i, j, mesh_theta >= 1.5 * np.pi]

    # set boundary condition
    F[:, 0, (mesh_theta <= np.pi)] = fn_b(mesh_x)[:, None]
    F[:, -1, (mesh_theta > np.pi)] = fn_t(mesh_x)[:, None]
    F[0, :, (mesh_theta <= 0.5 * np.pi) | (mesh_theta >= 1.5 * np.pi)] = fn_l(
        mesh_y
    )[None, :]
    F[-1, :, (mesh_theta > 0.5 * np.pi) & (mesh_theta < 1.5 * np.pi)] = fn_r(
        mesh_y
    )[None, :]

    return F

In [10]:
# def dvm_solver(
#     tensor_f: np.array, tol: float = 1e-5, max_iter: int = 2**8
# ) -> np.array:
#     for i in range(max_iter):
#         F = source_iteration(tensor_f)
#         diff = jnp.linalg.norm(F - tensor_f)
#         print(f"difference = {diff}")
#         if diff < tol:
#             print(f"converged at iter {i}")
#             break
#         tensor_f = F
#     return F

In [11]:
# tensor_f = dvm_solver(tensor_f)

In [ ]:
max_iter = 2**9
tol = 1e-5
for i in range(max_iter):
    F = source_iteration(tensor_f)
    diff = jnp.linalg.norm(F - tensor_f)
    print(f"difference = {diff}")
    if diff < tol:
        print(f"converged at iter {i}")
        break
    tensor_f = F
    

In [13]:
X, Y = np.meshgrid(mesh_x, mesh_y)

avg_f = 0.5 / np.pi * tensor_f @ weights[:, None]
avg_f = avg_f.squeeze()  # (num_x+1, num_y+1)

In [ ]:
font_format = {"family": "Times New Roman", "size": 20}

fig = plt.figure(figsize=(8, 6))
ax1 = fig.add_subplot(1, 1, 1, projection="3d")
surf1 = ax1.plot_surface(
    X, Y, avg_f, cmap="plasma", linewidth=0, antialiased=False
)
ax1.set_xlabel(r"$x$", font_format)
ax1.set_ylabel(r"$y$", font_format)
ax1.zaxis.set_major_locator(LinearLocator(5))
ax1.zaxis.set_major_formatter("{x:.01f}")
ax1.set_title("Reference solution", font_format)
ax1.spines["bottom"].set_linewidth(2)
ax1.spines["left"].set_linewidth(2)
ax1.spines["right"].set_linewidth(2)
ax1.spines["top"].set_linewidth(2)
fig.colorbar(surf1, shrink=0.5, aspect=5)

plt.show()


In [ ]:
font_format = {"family": "Times New Roman", "size": 20}

plt.figure(figsize=(8, 6))
ax1 = plt.subplot(1, 1, 1)
plt.contourf(X, Y, avg_f, 100, cmap="plasma")
# font size and type
plt.xticks(fontproperties="Times New Roman", size=12)
plt.yticks(fontproperties="Times New Roman", size=12)
plt.xlabel(r"$x$", font_format)
plt.ylabel(r"$v$", font_format)
plt.colorbar()
plt.title("Reference solution", font_format)
# # line width of four axises
ax1.spines["bottom"].set_linewidth(2)
ax1.spines["left"].set_linewidth(2)
ax1.spines["right"].set_linewidth(2)
ax1.spines["top"].set_linewidth(2)

plt.show()
plt.close()


In [ ]:
# np.savez(
#     f"../data/rte2d_ex6_ref_{kn:.1e}.npz",
#     mesh_x=mesh_x,
#     mesh_y=mesh_y,
#     rho=avg_f,
# )
